# MoE Unfrozen — **continue** main R0 run (`moe_unfrozen_sym_t14_v1`)

Resume training for the **primary thesis run** (clean-trained, 4 experts, top-k=1, unfrozen backbone).

**Before you run**
1. Runtime → **GPU** (T4 16 GB: use batch 64 if OOM; A100: batch 128).
2. Put your **run folder** on Google Drive (zip or unpacked). It must contain `options-and-config.pickle` and `checkpoints/*.pyt`.
3. Kaggle API token for COCO download (`kaggle.json`), unless COCO is already on Drive.
4. Colab disk ~100 GB for full COCO.

**Typical local path (zip before upload):**  
`results/experiments/unfrozen_moe/coco100k/4exp_k1/batch128/sym_t14_unfrozen_bal004warm10_ep20_2026-06-01`

`continue` loads the **latest** checkpoint in the folder. Set `RESUME_FROM_EPOCH` below to trim newer checkpoints first.

## 1. Configuration — edit these

In [ ]:
# --- Colab paths ---
PROJECT_ROOT = "/content/newmethod"
MOE_DIR = f"{PROJECT_ROOT}/hidden_moe_unfrozen"
RUNS_DIR = f"{MOE_DIR}/runs"
DATA_DIR = "/content/coco100k"
EXTRACT_DIR = "/content/coco_extract"

# --- git ---
REPO_URL = "https://github.com/ademladhari/newmethod.git"
REPO_BRANCH = "main"

# --- Kaggle (COCO download) ---
KAGGLE_USERNAME = ""
KAGGLE_KEY = ""

# --- Google Drive: run folder for continue ---
USE_GOOGLE_DRIVE = True
DRIVE_DATA_DIR = "/content/drive/MyDrive/coco100k"
# Option A: zip of the run folder (recommended)
DRIVE_RUN_ZIP = "/content/drive/MyDrive/moe_runs/sym_t14_unfrozen_bal004warm10_ep20_run.zip"
# Option B: unpacked run folder (if zip is empty, this is used)
DRIVE_RUN_FOLDER = "/content/drive/MyDrive/moe_runs/sym_t14_unfrozen_bal004warm10_ep20_2026-06-01"
# Option C: manual path after restore (leave "" to auto-detect)
RUN_FOLDER = ""

# Glob fallback if RUN_FOLDER empty after restore
EXPERIMENT_NAME = "moe_unfrozen_sym_t14_v1"

# --- continue settings ---
CONTINUE_EPOCHS = 60          # new total epoch count (was 20 at `new`)
RESUME_FROM_EPOCH = None      # e.g. 20 to delete checkpoints > 20; None = latest ckpt
NUM_WORKERS = 2               # Colab T4: 2; raise on A100 if stable
PREFETCH_FACTOR = 2
SAVE_EVERY = 1
PRINT_EACH = 100

## 2. GPU check + mount Google Drive

In [ ]:
import os
import shutil
import subprocess

import torch

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)
subprocess.run(["nvidia-smi"], check=False)

## 3. Kaggle credentials (COCO)

In [ ]:
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

if KAGGLE_USERNAME and KAGGLE_KEY:
    kaggle_json = kaggle_dir / "kaggle.json"
    kaggle_json.write_text(
        '{{"username":"{}","key":"{}"}}\n'.format(KAGGLE_USERNAME, KAGGLE_KEY)
    )
    os.chmod(kaggle_json, 0o600)
    print("Wrote", kaggle_json)
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            (kaggle_dir / "kaggle.json").write_bytes(uploaded["kaggle.json"])
            os.chmod(kaggle_dir / "kaggle.json", 0o600)
            print("Uploaded kaggle.json")
    except Exception as e:
        print("Set KAGGLE_USERNAME/KAGGLE_KEY or upload kaggle.json:", e)

need_kaggle = not (
    USE_GOOGLE_DRIVE
    and Path(DRIVE_DATA_DIR).is_dir()
    and (Path(DRIVE_DATA_DIR) / "train").exists()
)
if need_kaggle:
    assert (kaggle_dir / "kaggle.json").is_file(), "Missing ~/.kaggle/kaggle.json"
else:
    print("Skipping kaggle.json check — using COCO from Drive")

## 4. Clone repo + install deps

In [ ]:
import sys

if os.path.isdir(PROJECT_ROOT):
    !rm -rf {PROJECT_ROOT}

!git clone --branch {REPO_BRANCH} {REPO_URL} {PROJECT_ROOT}
%cd {MOE_DIR}
!pip -q install kagglehub==0.3.12 kaggle

sys.path.insert(0, MOE_DIR)
print("Working dir:", os.getcwd())

## 5. COCO 2017 → `train/` and `val/`

In [ ]:
import zipfile
from pathlib import Path

train_dst = Path(DATA_DIR) / "train"
val_dst = Path(DATA_DIR) / "val"
marker = Path(DATA_DIR) / ".ready"


def count_jpg(folder: Path) -> int:
    if not (folder.is_dir() or folder.is_symlink()):
        return 0
    return sum(1 for f in folder.iterdir() if f.suffix.lower() == ".jpg")


def find_image_dirs(root: Path):
    train_dir = val_dir = None
    for dirpath, _, filenames in os.walk(root):
        if not filenames:
            continue
        base = Path(dirpath).name.lower()
        if base == "train2017" and any(f.lower().endswith(".jpg") for f in filenames):
            train_dir = Path(dirpath)
        if base in ("val2017", "valid2017") and any(f.lower().endswith(".jpg") for f in filenames):
            val_dir = Path(dirpath)
    return train_dir, val_dir


def setup_train_val_symlinks(train_src: Path, val_src: Path):
    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    for dst in (train_dst, val_dst):
        if dst.is_symlink():
            dst.unlink()
        elif dst.is_dir():
            shutil.rmtree(dst)
    os.symlink(train_src.resolve(), train_dst, target_is_directory=True)
    os.symlink(val_src.resolve(), val_dst, target_is_directory=True)
    marker.write_text("ok\n")


if USE_GOOGLE_DRIVE and Path(DRIVE_DATA_DIR).is_dir() and count_jpg(Path(DRIVE_DATA_DIR) / "train") > 1000:
    print("Using COCO from Drive:", DRIVE_DATA_DIR)
    setup_train_val_symlinks(Path(DRIVE_DATA_DIR) / "train", Path(DRIVE_DATA_DIR) / "val")
elif marker.is_file() and count_jpg(train_dst) > 100000 and count_jpg(val_dst) > 4000:
    print("COCO already at", DATA_DIR)
else:
    import kagglehub
    cache_path = Path(kagglehub.dataset_download("awsaf49/coco-2017-dataset"))
    train_src, val_src = find_image_dirs(cache_path)
    if train_src is None or val_src is None:
        archive = next(cache_path.rglob("*.archive"), None)
        assert archive, "No .archive in kagglehub cache"
        extract_root = Path(EXTRACT_DIR)
        if not (extract_root / ".extract_done").is_file():
            extract_root.mkdir(parents=True, exist_ok=True)
            print("Extracting to", extract_root)
            with zipfile.ZipFile(archive, "r") as zf:
                zf.extractall(extract_root)
            (extract_root / ".extract_done").write_text("ok\n")
        train_src, val_src = find_image_dirs(extract_root)
    assert train_src and val_src
    setup_train_val_symlinks(train_src, val_src)

print("train images:", count_jpg(train_dst))
print("val images:", count_jpg(val_dst))

## 6. Restore run folder from Drive (required for `continue`)

In [ ]:
import zipfile
from glob import glob

os.makedirs(RUNS_DIR, exist_ok=True)

zip_path = Path(DRIVE_RUN_ZIP) if USE_GOOGLE_DRIVE else None
folder_path = Path(DRIVE_RUN_FOLDER) if USE_GOOGLE_DRIVE else None

if RUN_FOLDER and Path(RUN_FOLDER).is_dir():
    print("Using RUN_FOLDER:", RUN_FOLDER)
elif zip_path and zip_path.is_file():
    print("Unzipping run:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(RUNS_DIR)
elif folder_path and folder_path.is_dir():
    dest = Path(RUNS_DIR) / folder_path.name
    if dest.exists():
        shutil.rmtree(dest)
    print("Copying run folder:", folder_path, "->", dest)
    shutil.copytree(folder_path, dest)
else:
    existing = sorted(glob(f"{RUNS_DIR}/{EXPERIMENT_NAME}*"))
    if existing:
        print("Found existing run under runs/:", existing[-1])
    else:
        raise FileNotFoundError(
            "No run folder found. Upload zip to DRIVE_RUN_ZIP or set DRIVE_RUN_FOLDER on Drive."
        )

if not RUN_FOLDER:
    candidates = sorted(glob(f"{RUNS_DIR}/{EXPERIMENT_NAME}*"))
    if not candidates:
        candidates = sorted(glob(f"{RUNS_DIR}/sym_t14_unfrozen*"))
    assert candidates, f"No run under {RUNS_DIR}"
    RUN_FOLDER = candidates[-1]

RUN_FOLDER = str(Path(RUN_FOLDER).resolve())
opts = Path(RUN_FOLDER) / "options-and-config.pickle"
chk = list((Path(RUN_FOLDER) / "checkpoints").glob("*.pyt"))
print("RUN_FOLDER:", RUN_FOLDER)
print("options:", opts.is_file())
print("checkpoints:", len(chk), sorted(p.name for p in chk)[-3:])

## 7. (Optional) Trim checkpoints — resume from a specific epoch

In [ ]:
if RESUME_FROM_EPOCH is not None:
    trim_cmd = [
        sys.executable,
        f"{PROJECT_ROOT}/scripts/prepare_resume_from_epoch.py",
        "--run-folder", RUN_FOLDER,
        "--epoch", str(RESUME_FROM_EPOCH),
    ]
    print(" ".join(trim_cmd))
    subprocess.run(trim_cmd, check=True)
else:
    print("RESUME_FROM_EPOCH is None — continuing from latest checkpoint")

## 8. Continue training (main R0 run)

In [ ]:
import sys

os.chdir(MOE_DIR)

cont_cmd = [
    sys.executable, "-u", "train_moe.py", "continue",
    "--folder", RUN_FOLDER,
    "--data-dir", DATA_DIR,
    "--epochs", str(CONTINUE_EPOCHS),
    "--num-workers", str(NUM_WORKERS),
    "--pin-memory",
    "--prefetch-factor", str(PREFETCH_FACTOR),
    "--save-every", str(SAVE_EVERY),
    "--print-each", str(PRINT_EACH),
]
print(" ".join(cont_cmd))
subprocess.run(cont_cmd, check=True, cwd=MOE_DIR)

## 9. Save updated run to Google Drive

In [ ]:
from pathlib import Path

run_name = Path(RUN_FOLDER).name
zip_path = f"/content/{run_name}_continued.zip"

for name in ["train.csv", "validation.csv", "validation_noisy.csv"]:
    p = Path(RUN_FOLDER) / name
    if p.is_file():
        print(f"\n=== {name} (last rows) ===")
        subprocess.run(["tail", "-n", "3", str(p)], check=False)

if USE_GOOGLE_DRIVE:
    dest_root = "/content/drive/MyDrive/moe_runs"
    os.makedirs(dest_root, exist_ok=True)
    dest = f"{dest_root}/{run_name}"
    if os.path.isdir(dest):
        shutil.rmtree(dest)
    shutil.copytree(RUN_FOLDER, dest)
    print("Copied to", dest)
else:
    subprocess.run(["zip", "-r", zip_path, run_name], check=True, cwd=RUNS_DIR)
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        print("Download from file browser:", zip_path)